In [ ]:
import json
import os
from concurrent.futures import ThreadPoolExecutor, as_completed

In [ ]:
repo_root = os.path.dirname(os.path.dirname(os.path.dirname(os.getcwd())))
config_dir = os.path.join(repo_root, "config", "data_ingest")

with open(os.path.join(config_dir, "catalog_config.json")) as f:
    catalog = json.load(f)["catalog"]

daily_batch_config_path = os.path.join(config_dir, "daily_batch_config.json")
with open(daily_batch_config_path) as f:
    batch_config = json.load(f)

triggers = batch_config["triggers"]

In [ ]:
def run_worker(trigger):
    result = dbutils.notebook.run(
        "./worker",
        600,
        {
            "ticker": trigger["ticker"],
            "start_date": trigger["start_date"] or "",
            "catalog": catalog,
        },
    )
    return trigger["ticker"], result

results = {}
with ThreadPoolExecutor(max_workers=min(len(triggers), 8)) as executor:
    futures = [executor.submit(run_worker, trigger) for trigger in triggers]
    for future in as_completed(futures):
        ticker, result = future.result()
        results[ticker] = json.loads(result)

In [ ]:
for trigger in triggers:
    result = results.get(trigger["ticker"])
    if result and result["ingestionStatus"] == 1:
        trigger["start_date"] = result["last_date"]

In [ ]:
with open(daily_batch_config_path, "w") as f:
    json.dump(batch_config, f, indent=2)